# Proyecto 1 — Monitoreo transaccional: detectar lo que el orden revela
Integrantes: Iris Ayala, Anggie Quezada

Dataset: Sparkov (Credit Card Transactions Fraud Detection)

In [1]:
import sys
sys.path.append('src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datos import cargar_datos, split_temporal, construir_agregados, preparar_features_evento, construir_secuencias
from modelos import (entrenar_modelo_A, predecir_modelo_A, entrenar_modelo_B, predecir_modelo_B,
                     evaluar, prueba_permutacion, analisis_costo)

SEMILLA = 42
np.random.seed(SEMILLA)

## 1. Integridad de datos
Origen, tamaño, tasa de fraude, split temporal.

In [2]:
df = cargar_datos('data/fraudTrain.csv')
print(f"Total transacciones: {len(df)}")
print(f"Tasa de fraude: {df['is_fraud'].mean():.4%}")
print(f"Tarjetas únicas: {df['cc_num'].nunique()}")

train_raw, val_raw, test_raw = split_temporal(df)

Total transacciones: 1296675
Tasa de fraude: 0.5789%
Tarjetas únicas: 983
train: 778005 (2019-01-01 00:00:18 -> 2019-11-29 19:37:11)
val:   259335 (2019-11-29 19:37:13 -> 2020-03-06 07:15:17)
test:  259335 (2020-03-06 07:16:43 -> 2020-06-21 12:13:37)


In [3]:
# Features de evento (para secuencias) -- el diccionario de categorías se arma con train y se reutiliza
train_ev, cat2cod = preparar_features_evento(train_raw)
val_ev, _ = preparar_features_evento(val_raw, cat2cod=cat2cod)
test_ev, _ = preparar_features_evento(test_raw, cat2cod=cat2cod)

# Agregados (para modelo A)
train_agg = construir_agregados(train_ev)
val_agg = construir_agregados(val_ev)
test_agg = construir_agregados(test_ev)

COLUMNAS_AGREGADAS = ['monto_prom_24h', 'n_tx_ultima_hora', 'monto_max_dia', 'diversidad_comercio']

for d in (train_agg, val_agg, test_agg):
    d[COLUMNAS_AGREGADAS] = d[COLUMNAS_AGREGADAS].fillna(0)

## 2. Núcleo común: A vs B

In [4]:
# --- Modelo A: línea base sin orden ---
X_train_A = train_agg[COLUMNAS_AGREGADAS].values
y_train_A = train_agg['is_fraud'].values
X_test_A = test_agg[COLUMNAS_AGREGADAS].values
y_test_A = test_agg['is_fraud'].values

modelo_A, scaler_A = entrenar_modelo_A(X_train_A, y_train_A)
scores_A = predecir_modelo_A(modelo_A, scaler_A, X_test_A)
metricas_A = evaluar(y_test_A, scores_A)
print('Modelo A:', metricas_A)

Modelo A: {'auc_pr': np.float64(0.7260140137409181), 'f1': 0.6989409984871406, 'precision': 0.8354430379746836, 'recall': 0.6007802340702211}


In [5]:
# --- Modelo B: secuencial ---
LONGITUD_SECUENCIA = 10  # <- decisión a documentar en el README (probar 5/10/20 y justificar)

sec_train, y_seq_train, _ = construir_secuencias(train_ev, longitud=LONGITUD_SECUENCIA)
sec_val, y_seq_val, _ = construir_secuencias(val_ev, longitud=LONGITUD_SECUENCIA)
sec_test, y_seq_test, _ = construir_secuencias(test_ev, longitud=LONGITUD_SECUENCIA)

modelo_B = entrenar_modelo_B(sec_train, y_seq_train, sec_val, y_seq_val,
                              n_features=sec_train.shape[2])

scores_B = predecir_modelo_B(modelo_B, sec_test)
metricas_B = evaluar(y_seq_test, scores_B)
print('Modelo B:', metricas_B)

Época 1/10 - pérdida train: 0.0265 - pérdida val: 0.0076
Época 2/10 - pérdida train: 0.0070 - pérdida val: 0.0061
Época 3/10 - pérdida train: 0.0057 - pérdida val: 0.0051
Época 4/10 - pérdida train: 0.0052 - pérdida val: 0.0046
Época 5/10 - pérdida train: 0.0047 - pérdida val: 0.0045
Época 6/10 - pérdida train: 0.0045 - pérdida val: 0.0036
Época 7/10 - pérdida train: 0.0042 - pérdida val: 0.0043
Época 8/10 - pérdida train: 0.0041 - pérdida val: 0.0038
Época 9/10 - pérdida train: 0.0039 - pérdida val: 0.0041
Época 10/10 - pérdida train: 0.0038 - pérdida val: 0.0036
Modelo B: {'auc_pr': np.float64(0.9538924692897344), 'f1': 0.8997214484679665, 'precision': 0.9685157421289355, 'recall': 0.8400520156046815}


## 3. Valor del orden: pruebas de falsificación

In [6]:
# Prueba obligatoria 1: permutación controlada
auc_original, auc_barajada = prueba_permutacion(modelo_B, sec_test, y_seq_test)

# Interpretar honestamente:
# - si auc_original >> auc_barajada -> el orden SÍ aporta señal
# - si son similares -> no hay evidencia de que el orden aporte, decirlo así

In [7]:
# Prueba 2: quitar variables temporales
COLUMNAS_TEMPORALES = ["hora_del_dia", "dias_desde_ultima_tx"]

test_ev_sin_tiempo = test_ev.copy()
test_ev_sin_tiempo[COLUMNAS_TEMPORALES] = 0

sec_test_sin_tiempo, y_test_sin_tiempo, _ = construir_secuencias(test_ev_sin_tiempo, longitud=LONGITUD_SECUENCIA)
scores_sin_tiempo = predecir_modelo_B(modelo_B, sec_test_sin_tiempo)
metricas_sin_tiempo = evaluar(y_test_sin_tiempo, scores_sin_tiempo)

print("con variables temporales:", metricas_B)
print("sin variables temporales:", metricas_sin_tiempo)

con variables temporales: {'auc_pr': np.float64(0.9538924692897344), 'f1': 0.8997214484679665, 'precision': 0.9685157421289355, 'recall': 0.8400520156046815}
sin variables temporales: {'auc_pr': np.float64(0.4675552107012779), 'f1': 0.347726948838535, 'precision': 0.2226683701405366, 'recall': 0.7932379713914174}


Al quitar las variables temporales (hora del día y días desde la última transacción), el desempeño del modelo B cae de forma drástica: el AUC-PR baja de 0.954 a 0.468, y el F1 pasa de 0.90 a 0.35. Aunque la exhaustividad se mantiene relativamente alta (0.79 contra 0.84 original), la precisión se desploma de 0.97 a 0.22, lo que significa que el modelo empieza a marcar como fraude una enorme cantidad de transacciones legítimas cuando pierde la información de ritmo temporal. Esto indica que buena parte de lo que el modelo B estaba aprovechando no es el orden puro de montos y categorías, sino el patrón de *cuándo* ocurren las transacciones entre sí — es decir, gran parte de la señal que capturaba B dependía fuertemente del componente temporal, y no exclusivamente de la secuencia de eventos en sí misma. El orden aporta valor, pero mezclado con el ritmo temporal, y no queda tan claro cuánto de esa ganancia es "orden puro" versus "información de tiempo entre eventos".